In [1]:
import os
import sys

# to setup import paths add project root dirs to sys.path
sys.path.append(os.path.join(os.getcwd(), "..", ".."))
from baseVR.base_functionality import init_import_paths
init_import_paths()

In [2]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# %matplotlib qt
%matplotlib widget

from analytics_processing import analytics
import analytics_processing.analytics_constants as C
from CustomLogger import CustomLogger as Logger

from dashsrc.plot_components.plot_wrappers.data_selection import group_filter_data

from analytics_processing.modality_loading import session_modality_from_nas
from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args
from analytics_processing.sessions_from_nas_parsing import fullfnames2snames

# import GridSpecFromSubplotSpec
from matplotlib.gridspec import GridSpecFromSubplotSpec
import scipy.stats as stats
import statsmodels.api as sm
from matplotlib.colors import LinearSegmentedColormap, Normalize



In [3]:
output_dir = "./outputs/experimental/"
data = {}
nas_dir = C.device_paths()[0]
Logger().init_logger(None, None, logging_level="WARNING")

In [4]:
animal_ids = [6]
paradigm_ids = [1100]
session_ids = None

In [5]:
session_dirs, _ = sessionlist_fullfnames_from_args(paradigm_ids, animal_ids, session_ids)
session_names = fullfnames2snames(session_dirs)

In [6]:
fr_raw = analytics.get_analytics('FiringRate40msHz', session_names=session_names)
fr = fr_raw.drop("from_ephys_timestamp", axis=1)
fr.index = fr.index.droplevel((0,1,3, ))
fr.set_index('to_ephys_timestamp', append=True, inplace=True)
fr['global_t'] = np.arange(40_000, 40_000*len(fr)+1, 40_000)
fr.set_index('global_t', append=True, inplace=True)
fr.index = fr.index.rename(['session_id', 'session_t', 'global_t'])
session_t_start = fr.index[fr.index.get_level_values('session_t') == 40_000]

fr = fr.fillna(0)
fr_z = fr.apply(lambda unit_fr: ((unit_fr - unit_fr.mean()) / unit_fr.std()))

fr

Unit0001  Unit0002  Unit0003  Unit0004  \
session_id session_t  global_t                                              
1          40000      40000               0         0         0         0   
           80000      80000               0         0         0         0   
           120000     120000              0         0         0         0   
           160000     160000              0         0         0         0   
           200000     200000              0         0         0         0   
...                                     ...       ...       ...       ...   
33         4435960000 53320480000         0         0         0         0   
           4436000000 53320520000         0         0         0         0   
           4436040000 53320560000         0         0         0         0   
           4436080000 53320600000         0         0         0         0   
           4436120000 53320640000         0         0         0         0   

                                   Unit0005  Unit0006  Unit0007  Unit0008  \
session_id session_t  global_t                                              
1          40000      40000               0         0         0         0   
           80000      80000               0         0         0         0   
           120000     120000              0         0         0         0   
           160000     160000              0         0         0         0   
           200000     200000              0         0         0         0   
...                                     ...       ...       ...       ...   
33         4435960000 53320480000         0         0         0         0   
           4436000000 53320520000         0         0         0         0   
           4436040000 53320560000         0         0         0         0   
           4436080000 53320600000         0         0         0         0   
           4436120000 53320640000         0         0         0         0   

                                   Unit0009  Unit0010  ...  Unit0068  \
session_id session_t  global_t                         ...             
1          40000      40000               0         0  ...         0   
           80000      80000               0         0  ...         0   
           120000     120000              0         0  ...         0   
           160000     160000              0         0  ...         0   
           200000     200000              0         0  ...         0   
...                                     ...       ...  ...       ...   
33         4435960000 53320480000         0         0  ...        25   
           4436000000 53320520000         0         0  ...        25   
           4436040000 53320560000         0         0  ...        25   
           4436080000 53320600000         0         0  ...        25   
           4436120000 53320640000         0         0  ...         0   

                                   Unit0069  Unit0070  Unit0071  Unit0072  \
session_id session_t  global_t                                              
1          40000      40000               0         0         0         0   
           80000      80000               0         0         0         0   
           120000     120000              0         0         0         0   
           160000     160000              0         0         0         0   
           200000     200000              0         0         0         0   
...                                     ...       ...       ...       ...   
33         4435960000 53320480000         0         0        25         0   
           4436000000 53320520000         0         0        25         0   
           4436040000 53320560000        25         0        25         0   
           4436080000 53320600000         0         0         0         0   
           4436120000 53320640000         0        25         0         0   

                                   Unit0073  Unit0074  Unit0075  Unit0076  \
session_id sess

In [7]:
behavior_glm_input = analytics.get_analytics('Behavior40msAligned', session_names=session_names)
# drop index level
behavior_glm_input.index = behavior_glm_input.index.droplevel(['entry_id', 'paradigm_id', 'animal_id'])
drop_cols = ['trial_id',
             'cue',
             'trial_outcome',
             'choice_R1',
             'choice_R2',
             'trial_start_pc_timestamp',
             'from_ephys_timestamp',
             'to_ephys_timestamp',]
behavior_glm_input.columns

Index(['trial_id', 'cue', 'trial_outcome', 'choice_R1', 'choice_R2',
       'trial_start_pc_timestamp', 'from_ephys_timestamp',
       'to_ephys_timestamp', 'frame_velocity', 'frame_raw',
       'frame_raw_quantile', 'frame_acceleration', 'abs_frame_acceleration',
       'frame_positive_acceleration', 'frame_negative_acceleration',
       'frame_yaw', 'frame_yaw_left', 'frame_yaw_right', 'frame_pitch',
       'frame_pitch_left', 'frame_pitch_right', 'lick_count', 'post_lick',
       'post_reward_sound', 'post_reward', 'frame_position', 'visible_cue',
       'zone_before_reward1', 'zone_before_reward2', 'zone_between_cues',
       'zone_cue2', 'zone_cue2_passed', 'zone_cue2_visible',
       'zone_post_reward', 'zone_reward1', 'zone_reward2', 'head_angle',
       'head_angle_left', 'head_angle_right', 'head_angle_velocity',
       'head_angle_velocity_left', 'head_angle_velocity_right'],
      dtype='object')

In [ ]:
# ==============================================================
# === COMPUTATION UTILITIES ====================================
# ==============================================================

def compute_distribution_params(unit_hz):
    """
    Compute firing rate statistics and PMFs for Poisson and Negative Binomial.
    Returns dict with mean, std, variance, PMFs, etc.
    """
    m = unit_hz.mean()
    std = unit_hz.std()
    var = std ** 2

    # Convert firing rates to discrete counts
    discrete_counts = np.round(unit_hz.values).astype(int)
    discrete_counts = discrete_counts[discrete_counts >= 0]
    lam = discrete_counts.mean()

    # PMFs
    x = np.arange(0, max(10, discrete_counts.max()) + 1)
    pois_pmf = stats.poisson.pmf(x, lam)

    # Negative binomial params
    p = m / var if var > m else 0.999  # prevent division by zero
    r = m * p / (1 - p)
    nb_pmf = stats.nbinom.pmf(x, r, p)

    return {
        "mean": m,
        "std": std,
        "var": var,
        "lam": lam,
        "x": x,
        "pois_pmf": pois_pmf,
        "nb_pmf": nb_pmf,
    }


In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# from matplotlib.ticker import MultipleLocator
# from scipy import stats


# # ==============================================================
# # === PLOTTING UTILITIES =======================================
# # ==============================================================

# def plot_firing_rate_trace(ax, unit_hz, session_t_start, fr_index, neuron_i):
#     """Plot main firing rate trace with session demarcations."""
#     ax.plot(fr_index.get_level_values('global_t') / (1e6 * 60), unit_hz, alpha=0.7)
#     for i in range(len(session_t_start)):
#         s_id, _, global_t_start = session_t_start[i]
#         global_t_end = (
#             session_t_start[i + 1][2]
#             if i + 1 < len(session_t_start)
#             else fr_index.get_level_values('global_t').max()
#         )
#         avg_fr = unit_hz[
#             (fr_index.get_level_values('global_t') >= global_t_start)
#             & (fr_index.get_level_values('global_t') < global_t_end)
#         ].mean()
#         s_str = f"S{s_id:02}\nAvg {avg_fr:.1f}Hz"
#         ax.axvline(global_t_start / (1e6 * 60), color='gray', linestyle='--', alpha=0.5)
#         ax.text(global_t_start / (1e6 * 60) + 0.1, ax.get_ylim()[1] * 0.9, s_str,
#                 verticalalignment='top', color='gray', alpha=0.7)

#     ax.spines['top'].set_visible(False)
#     ax.spines['right'].set_visible(False)
#     ax.set_title(f'Neuron {neuron_i}')
#     ax.set_xlabel('Time (minutes)')
#     ax.set_ylabel('Firing Rate (Hz)')
#     ax.yaxis.set_major_locator(MultipleLocator(25))
#     ax.grid(axis='y', linestyle='--', alpha=0.5)


# def plot_histograms(ax1, ax2, unit_hz, x, nb_pmf, pois_pmf):
#     """Plot histogram + Poisson and NegBinom fits."""
#     # First histogram (log x-scale)
#     ax1.hist(unit_hz, bins=20, orientation='horizontal', color='gray', alpha=0.7, density=True)
#     ax1.plot(nb_pmf, x, 'b--', lw=2)
#     ax1.plot(np.clip(pois_pmf, 1e-22, None), x, 'r--', lw=2)
#     ax1.set_xscale('log')
#     ax1.set_xlabel('Count')
#     ax1.set_ylabel('Firing Rate (Hz)')
#     ax1.yaxis.set_major_locator(MultipleLocator(25))
#     ax1.grid(axis='y', linestyle='--', alpha=0.5)

#     # Second histogram (linear x-scale)
#     bins = ax2.hist(unit_hz, bins=20, orientation='horizontal', color='gray', alpha=0.7, density=True)[0]
#     ax2.plot(nb_pmf, x, 'b--', lw=2, label='NegBinom')
#     ax2.plot(pois_pmf, x, 'r--', lw=2, label='Poisson')
#     ax2.set_xlabel('Density')
#     ax2.set_ylabel('Firing Rate (Hz)')
#     ax2.legend()
#     ax2.yaxis.set_major_locator(MultipleLocator(25))
#     ax2.set_xlim(0, np.max(bins) * 1.2)
#     ax2.grid(axis='y', linestyle='--', alpha=0.5)


# # ==============================================================
# # === MAIN WRAPPER =============================================
# # ==============================================================

# def plot_neuron_summary(fr, session_t_start, neuron_indices=None):
#     """
#     For each neuron, plot firing rate trace and histogram with Poisson/NB fits.
#     """
#     plt.close('all')
#     if neuron_indices is None:
#         neuron_indices = range(fr.shape[1])

#     all_params = []
#     for neuron_i in neuron_indices:
#         fig, ax = plt.subplots(
#             ncols=3, nrows=1, figsize=(40, 3), sharey=False,
#             tight_layout=True, width_ratios=(25, 1, 1)
#         )
#         fig.subplots_adjust(left=0.05, right=0.95)

#         unit_hz = fr.iloc[:, neuron_i]
#         params = compute_distribution_params(unit_hz)
#         all_params.append(params)

#         plot_firing_rate_trace(ax[0], unit_hz, session_t_start, fr.index, neuron_i)
#         plot_histograms(ax[1], ax[2], unit_hz, params["x"], params["nb_pmf"], params["pois_pmf"])

#         # Display summary text
#         m, std = params["mean"], params["std"]
#         ax[2].set_title(f'Mean: {m:.2f}\nSTD: {std:.2f}')

#         if neuron_i > 1:
#             break

#     plt.show()
#     return all_params

# # ==============================================================
# # === EXAMPLE USAGE ============================================
# # ==============================================================


# # Example call (assuming `fr` and `session_t_start` are defined):
# all_params = plot_neuron_summary(fr, session_t_start)
# # or for a subset:
# # plot_neuron_summary(fr, session_t_start, neuron_indices=range(5))


In [ ]:
# plt.close('all')
# fig, ax1 = plt.subplots()
# for i, param in enumerate(all_params):
#     x = param["x"]
#     nb_pmf = param["nb_pmf"]
#     cum_nb_pmf = np.cumsum(nb_pmf)
#     # 99% cutoff
#     cutoff_idx = np.where(cum_nb_pmf >= 0.99)[0][0]
#     nb_pmf = nb_pmf[:cutoff_idx+1]
#     x = x[:cutoff_idx+1]

#     # print(nb_pmf)
#     col = 'olive' if i <20 else 'pink'    
#     ax1.plot(x, nb_pmf, col, lw=2, alpha=0.6)
#     ax1.set_yscale('log')
#     ax1.set_ylabel('Density')
#     ax1.set_xlabel('Firing Rate (Hz)')
#     ax1.xaxis.set_major_locator(MultipleLocator(25))
#     ax1.grid(axis='x', linestyle='--', alpha=0.5)
#     ax1.tick_params(axis='y', which='major', labelleft=True)

# plt.show()

In [ ]:

def fr_colors(maxval=32):
    """Return (cmap, norm) for firing-rate heatmaps.

    - maxval: maximum FR value (same scale used when building stops).
    - cmap: matplotlib colormap (continuous, 256 colors).
    - norm: Normalize(vmin=0, vmax=maxval) to pass to imshow/Scatter/Colorbar.
    """
    eps = 1e-6
    stops = [
        (0, '#ffffff'),
        ((0 + eps) / maxval, "#e8e8e8"),
        (0.1 / maxval, "#e8e8e8"),
        ((0.1 + eps) / maxval, "#8C6565"),
        (1.0 / maxval, "#8C6565"),
        ((1 + eps) / maxval, "#8C6565"),
        (2.0 / maxval, "#8C6565"),
        ((2 + eps) / maxval, "#963F3F"),
        (4.0 / maxval, "#963F3F"),
        ((4 + eps) / maxval, "#A73131"),
        (8.0 / maxval, "#A73131"),
        ((8 + eps) / maxval, "#D1552C"),
        (16.0 / maxval, "#D1552C"),
        ((16 + eps) / maxval, "#D1922C"),
        (32.0 / maxval, "#D1922C"),
    ]

    # defensive: ensure positions are in [0,1] and sorted
    stops = [(min(max(float(p), 0.0), 1.0), c) for p, c in stops]
    stops = sorted(stops, key=lambda t: t[0])

    # build a continuous colormap from the stop list
    cmap = LinearSegmentedColormap.from_list("fr_cmap", stops, N=256)

    # normalization maps data in [0, maxval] to [0,1] for the colormap
    norm = Normalize(vmin=0.0, vmax=float(maxval))
    return cmap, norm


In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# from matplotlib.ticker import MultipleLocator, FormatStrFormatter
# from scipy import stats

# # ==============================================================
# # === COMPUTATION UTILITIES ====================================
# # ==============================================================
# import matplotlib.colors as mcolors

# def compute_distribution_params(unit_hz):
#     """Compute firing rate statistics and PMFs for Poisson and Negative Binomial."""
#     m = unit_hz.mean()
#     std = unit_hz.std()
#     var = std ** 2

#     # Convert firing rates to discrete counts
#     discrete_counts = np.round(unit_hz.values).astype(int)
#     discrete_counts = discrete_counts[discrete_counts >= 0]
#     lam = discrete_counts.mean()

#     # PMFs
#     x = np.arange(0, max(10, discrete_counts.max()) + 1)
#     pois_pmf = stats.poisson.pmf(x, lam)

#     # Negative binomial params
#     p = m / var if var > m else 0.999  # prevent division by zero
#     r = m * p / (1 - p)
#     nb_pmf = stats.nbinom.pmf(x, r, p)

#     return {
#         "mean": m,
#         "std": std,
#         "x": x,
#         "pois_pmf": pois_pmf,
#         "nb_pmf": nb_pmf,
#     }

# # ==============================================================
# # === GRID PLOTTING ============================================
# # ==============================================================

# def plot_neuron_session_histograms(fr, session_t_start, neuron_indices=None, n_bins=10):
#     """
#     Plot log(count histograms) for each neuron (rows) and session (columns).
#     Each subplot shows histogram of log firing rates and mean/std in title.
#     """
#     if neuron_indices is None:
#         neuron_indices = range(fr.shape[1])
    
#     session_names = [f"S{s_id:02}" for s_id, _, _ in session_t_start]
#     n_sessions = len(session_names)
#     n_neurons = len(neuron_indices)

#     fig, axes = plt.subplots(
#         nrows=n_neurons, ncols=n_sessions,
#         figsize=(1.4 * n_sessions, 2 * n_neurons),
#         sharex=True, sharey=True
#     )
    
#     cmap, norm = fr_colors(maxval=32)
    
#     # fig.colorbar(im, ax=ax, label='Firing rate (Hz)')

#     # Ensure 2D array of axes
#     if n_neurons == 1:
#         axes = np.expand_dims(axes, 0)
#     if n_sessions == 1:
#         axes = np.expand_dims(axes, 1)

#     fr_index = fr.index.get_level_values('global_t')

#     # Compute start-end time of each session
#     session_boundaries = []
#     for i, (s_id, _, global_t_start) in enumerate(session_t_start):
#         global_t_end = (
#             session_t_start[i + 1][2]
#             if i + 1 < len(session_t_start)
#             else fr_index.max()
#         )
#         session_boundaries.append((s_id, global_t_start, global_t_end))

#     # Iterate neurons × sessions
#     for row, neuron_i in enumerate(neuron_indices):
#         for col, (s_id, t_start, t_end) in enumerate(session_boundaries):
#             ax = axes[row, col]
#             unit_hz = fr.iloc[:, neuron_i]
#             session_mask = (fr_index >= t_start) & (fr_index < t_end)
#             session_data = unit_hz[session_mask]

#             if np.all(session_data == 0):
#                 ax.text(0.5, 0.5, "no spikes", ha='center', va='center', fontsize=8)
#                 # turn off spines
#                 [ax.spines[side].set_visible(False) for side in ['top', 'right', 'left', 'bottom']]
#                 ax.set_xticks([]); ax.set_yticks([])
#                 # axis off
#                 ax.axis('off')
#                 continue

#             params = compute_distribution_params(session_data)

#             x = params["x"]
#             nb_pmf = params["nb_pmf"]
#             pois_pmf = params["pois_pmf"]
#             ax.plot(x, nb_pmf, 'b--', lw=2, label='NegBinom')
#             ax.plot(np.clip(pois_pmf, 1e-22, None), x, 'r--', lw=2)

#             bin_edges = np.arange(0, 301, 25)  # 0–300 Hz in 25 Hz steps
#             ax.hist(session_data, bins=bin_edges, color='gray', alpha=0.6, density=True)

#             m, std = session_data.mean(), session_data.std()
#             # ax.set_title(f"{m:.2f} ± {std:.2f}", fontsize=8)

#             if row == n_neurons - 1:
#                 ax.set_xlabel(session_names[col], fontsize=9)
#             if col == 0:
#                 ax.set_ylabel(f"Neuron {neuron_i}", fontsize=9)
            
#             # draw a dot for firing rate
#             # get max min x y values
#             x_corner, y_corner = 250, 50
#             ax.scatter([x_corner], [y_corner], color=cmap(norm(m)), s=300, 
#                        edgecolor='k', linewidth=.5, zorder=3)
#             ax.text(x_corner-60, y_corner, f"{m:.2}Hz", ha='right', va='center',
#                     fontsize=8)
            
#             sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
            
#             # add thin vertical lines every 100, 200, 300 Hz
#             # [ax.axvline(x, linestyle='-', linewidth=.5, color='k', alpha=.2) 
#             #  for x in (100,200,300)]
            
#             # set ticks at the 25 Hz grid and format as integer labels
#             ax.xaxis.set_major_locator(MultipleLocator(25))
#             ax.xaxis.set_major_formatter(FormatStrFormatter('%d'))
#             plt.setp(ax.xaxis.get_ticklabels(), rotation=0, fontsize=8)
#             plt.setp(ax.yaxis.get_ticklabels(), rotation=0, fontsize=8)
#             ax.spines['top'].set_visible(False)
#             ax.spines['right'].set_visible(False)
#             ax.grid(axis='x', linestyle='--', alpha=0.4)
#             ax.grid(False, axis='y')
#             ax.set_yscale('log')
            
            

#     plt.tight_layout()
#     # fig.colorbar(sm, ax=ax, label='Firing rate (Hz)')

# # Example usage (unchanged)
# # set of 5 neurons 
# plt.close("all")
# for neuron_i in range(0,5,5):
#     plot_neuron_session_histograms(fr, session_t_start, neuron_indices=range(neuron_i,neuron_i+5))
#     plt.show()

In [ ]:
# # Verify results
# print("Original lick events:", behavior_glm_input['lick_count'].sum())
# print("Dilated lick events:", behavior_glm_input['lick_count_dilated'].sum())

# # Plot a small segment to visualize
# plt.figure(figsize=(15, 3))
# start_idx = 3000
# window = 200
# plt.plot(behavior_glm_input['lick_count'].iloc[start_idx:start_idx+window].values, 
#          label='Original', alpha=0.7)
# plt.plot(behavior_glm_input['lick_count_dilated'].iloc[start_idx:start_idx+window].values, 
#          label='Dilated', alpha=0.7)
# plt.legend()
# plt.title('Original vs Dilated Lick Events')
# plt.show()

In [ ]:
behavior_glm_input['choice_R1'] = behavior_glm_input['choice_R1'].astype(float)
behavior_glm_input['choice_R2'] = behavior_glm_input['choice_R2'].astype(float)
behavior_glm_input.iloc[1000]


In [ ]:
import statsmodels.api as sm
cmap, norm = fr_colors(maxval=32)
max_fr = 200

def add_joint_plot(fig, parent_spec, x, y, color='gray', alpha=0.01, share_x_with=None, share_y_with=None):
    """Creates a joint scatter plot (no marginals) inside a subplot grid cell."""
    # Convert to numpy arrays
    x_np = np.array(x, dtype=np.float64) # behavior variable, regresser
    y_np = np.array(y, dtype=np.float64) # firing rate
    
    # Create mask for valid values
    valid_mask = ~(np.isnan(x_np) | np.isnan(y_np))
    x_clean = x_np[valid_mask]
    y_clean = y_np[valid_mask]

    # Create subplot and share axes if requested
    gs = GridSpecFromSubplotSpec(1, 1, subplot_spec=parent_spec)
    ax_joint = fig.add_subplot(gs[0, 0], sharex=share_x_with, sharey=share_y_with)

    # Plot data, y the firing rate is on the x axis, x the behavior regresser is on the y axis
    # add jitter to x for better visualization
    jitter_strength = 0.1 * (np.max(y_clean) - np.min(y_clean))
    y_jittered = y_clean + np.random.uniform(-jitter_strength, jitter_strength, size=y_clean.shape)
    # create a sample mask, sample 2000 points
    mask = np.random.choice([False, True], size=x_clean.shape, p=[0.9, 0.1])
    ax_joint.scatter(y_jittered[mask], x_clean[mask], s=6, color=color, alpha=alpha)
    
    # jitter the other
    jitter_strength = 0.05 * (np.max(x_clean) - np.min(x_clean))
    x_jittered = x_clean + np.random.uniform(-jitter_strength, jitter_strength, size=x_clean.shape)
    ax_joint.scatter(y_jittered[mask], x_jittered[mask], s=6, color=color, alpha=alpha)

    # Optionally add GLM fit
    fit_glm = True
    if fit_glm:
        add_glm_fit(ax_joint, x_clean, y_clean)


    # Turn off ticks
    ax_joint.tick_params(labelleft=False, labelbottom=False)
    [ax_joint.spines[side].set_visible(False) for side in ['top', 'right', 'left', 'bottom']]
    
    return ax_joint

# ==============================================================
# === GLM FITTING AND VISUALIZATION ============================
# ==============================================================
import statsmodels.api as sm

def add_glm_fit(ax, x, y, family=sm.families.NegativeBinomial()):
    """
    Fit a GLM and a null model, plot both fits, and print summaries.

    Parameters
    ----------
    ax : matplotlib axis
        Axis to draw on.
    x, y : array-like
        Data.
    family : statsmodels.family instance
        GLM family, e.g. sm.families.NegativeBinomial().
    """
    # === Full model ===
    mu = np.mean(y)
    if mu < 0.1:
        return None, None  # skip very low firing rates
    var = np.var(y)
    alpha = (var - mu) / (mu * mu) if var > mu else 1.0
    family = sm.families.NegativeBinomial(alpha=alpha)
    
    X = sm.add_constant(x)
    model = sm.GLM(y, X, family=family)
    try:
        result_full = model.fit()
    except Exception as e:
        print(f"GLM fitting failed: {e}")
        return None, None
    # print("==== Full model ====")
    # print(result_full.summary())

    # Predictions for full model
    y_pred_full = result_full.predict(X)
    sort_idx = np.argsort(x)
    ax.plot(y_pred_full[sort_idx], x[sort_idx], color='red', lw=1.2, label='GLM fit')

    # === Null model (intercept only) ===
    X_null = np.ones_like(x)  # no slope term
    # X_null = np.ones((len(x), 1))
    model_null = sm.GLM(y, X_null, family=family)
    result_null = model_null.fit()
    # print("\n==== Null model ====")
    # print(result_null.summary())

    # Predictions for null model (constant mean)
    y_pred_null = result_null.predict(X_null)
    ax.plot(y_pred_null, x, color='blue', lw=1.0, linestyle='--', label='Null model')
    ax.set_xlim(0, max_fr+1)

    # === Model comparison ===
    from scipy.stats import chi2
    lr_stat = 2 * (result_full.llf - result_null.llf)
    df_diff = result_full.df_model - result_null.df_model
    p_value = chi2.sf(lr_stat, df_diff)  # Changed from chisqprob to chi2.sf

    if p_value < 0.05 and result_full.pseudo_rsquared() > .02:
        # draw in top right corner of ax
        print("Significant GLM fit: p =", p_value, result_full.summary())
        ax.text(0.95, 0.95, f"* {result_full.pseudo_rsquared():.3f}", transform=ax.transAxes,
                fontsize=16, color='red', ha='right', va='top')
    return result_full, result_null


def plot_neuron_session_histograms(fr, session_t_start, behavior_aligned, neuron_i=0, n_bins=10):
    session_names = [f"S{s_id:02}" for s_id, _, _ in session_t_start]
    n_sessions = len(session_names)
    n_regressers = behavior_aligned.shape[1]

    # Create figure with specific layout
    fig = plt.figure(figsize=(1.8 * n_sessions, 2.2 * n_regressers))
    
    # Create GridSpec with extra row/column for histograms
    outer_gs = plt.GridSpec(
        nrows=n_regressers + 1, 
        ncols=n_sessions + 1, 
        figure=fig,
        wspace=0.2, hspace=0.2,
        width_ratios=[0.5] + [1]*n_sessions,  # Make first column narrower
        height_ratios=[0.5] + [1]*n_regressers  # Make first row shorter
    )

    # === LEFT COLUMN: behavior histograms ===
    left_hist_axes = []
    for row, regr_name in enumerate(behavior_aligned.columns):
        print(regr_name)
        ax = fig.add_subplot(outer_gs[row + 1, 0])
        left_hist_axes.append(ax)
        regr_data = behavior_aligned[regr_name]
        
        # Calculate density for consistent y-limits across row
        # Remove NA/NaN values before plotting
        regr_data_clean = regr_data.dropna()
        counts, bins, _ = ax.hist(regr_data_clean, bins=20, density=True, 
                                orientation="horizontal", color='gray', alpha=0.6)
        
        ax.set_ylabel(f"{regr_name}", fontsize=13)
        ax.set_xscale('log')
        ax.grid(axis='x', linestyle='--', alpha=0.4)
        [ax.spines[side].set_visible(False) for side in ['top', 'bottom', 'right']]
        
        # Store density limits for sharing
        ax.density_limits = (min(counts[counts > 0]), max(counts))
        
        if row == n_regressers - 1:
            ax.tick_params(labelsize=7, labelbottom=True)
            ax.set_ylabel("Density", fontsize=8)
        else:
            plt.setp(ax.get_xticklabels(), visible=False)
            
        # Reverse x-axis for better visualization
        xlim = ax.get_xlim()
        ax.set_xlim(xlim[1], xlim[0])

    # === TOP ROW: neuron firing distribution ===
    top_row_axes = []
    max_density = 0  # Track maximum density for consistent scaling
    
    # First pass to get max density
    for s_id in fr.index.unique('session_id'):
        session_data = fr.iloc[:, neuron_i].loc[s_id].dropna()
        if len(session_data) > 0:
            counts, _ = np.histogram(session_data, bins=np.arange(0, max_fr+1, 25), density=True)
            max_density = max(max_density, max(counts))

    # Second pass to plot with consistent scaling
    for col, s_id in enumerate(fr.index.unique('session_id')):
        ax = fig.add_subplot(outer_gs[0, col + 1])
        top_row_axes.append(ax)
        
        session_data = fr.iloc[:, neuron_i].loc[s_id].dropna()
        if len(session_data) == 0 or session_data.mean() < .1:
            ax.axis('off')
            ax.set_title(f"S{s_id:02}\nno spikes", fontsize=8)
            continue

        counts, bins, _ = ax.hist(session_data, bins=np.arange(0, max_fr+1, 25), 
                                color='gray', alpha=0.6, density=True)
        
        params = compute_distribution_params(session_data)
        ax.plot(params["x"], params["nb_pmf"], 'b--', lw=1.5, label='NegBinom')
        ax.set_yscale('log')
        
        # Add firing rate indicator in top right corner using axis coordinates
        m = session_data.mean()
        ax.scatter(0.8, 0.8, color=cmap(norm(m)), s=300, 
                   transform=ax.transAxes,
                  edgecolor='k', linewidth=.5, zorder=3)
        ax.text(0.6, 0.8, f"{m:.2}Hz", ha='right', va='center',
                transform=ax.transAxes,
                fontsize=8)
        
        ax.set_title(session_names[col], fontsize=8)
        ax.grid(axis='x', linestyle='--', alpha=0.4)
        [ax.spines[side].set_visible(False) for side in ['top', 'right', 'left']]
        
        # Set consistent y-limits
        ax.set_ylim(0, max_density * 1.2)
        
        if col == 0:
            ax.set_ylabel(f"Density, Neuron {neuron_i}", fontsize=8)
        else:
            plt.setp(ax.get_yticklabels(), visible=False)
    
    # === MAIN GRID: joint plots ===
    # get min and max of behavior_aligned for normalization
    
    for row, regr_name in enumerate(behavior_aligned.columns):
        print("Fitting regresser:", regr_name, '... ')
        for col, s_id in enumerate(fr.index.unique('session_id')):
            regr_data = behavior_aligned[regr_name].loc[s_id].values
            unit_hz = fr.iloc[:, neuron_i].loc[s_id].values
            if (unit_hz.mean() < .1):
                continue
            
            _ = add_joint_plot(fig, outer_gs[row + 1, col + 1], 
                               regr_data, unit_hz,
                               share_x_with=top_row_axes[col],
                               share_y_with=left_hist_axes[row])
            # break
        # break
    fig.suptitle(f"Neuron {neuron_i} firing rate vs behavior regressers GLM", fontsize=16)
        
    # set left and right margin to 0.05
    plt.subplots_adjust(left=0.05, right=0.95)
    plt.savefig(f"../outputs/glm_results/neuron_{neuron_i}_GLM.png", dpi=300)
    plt.show()
    # plt.close()

plt.close("all")
# for neuron_i in range(0,77):
for neuron_i in range(1,2):
    plot_neuron_session_histograms(fr, session_t_start, behavior_glm_input, neuron_i=neuron_i)
    break

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import nbinom

sns.set(style="whitegrid", context="talk")

# ==============================================================
# === SIMULATE DATA ============================================
# ==============================================================

np.random.seed(42)
n = 200

# Predictive regressor X1: correlated with rate
X1 = np.linspace(-2, 2, n)
# Irrelevant regressor X2: random noise
X2 = np.random.randn(n)

# True model parameters for X1
beta0, beta1 = 1.0, 0.8

# Mean firing rate (Poisson rate λ or mean of NegBin)
mu_true = np.exp(beta0 + beta1 * X1)

# Dispersion parameter for negative binomial
r = 5.0  # larger r → closer to Poisson
p = r / (r + mu_true)

# Draw negative binomial counts
Y = nbinom(n=r, p=p).rvs()

# ==============================================================
# === FIT SIMPLE MODELS ========================================
# ==============================================================

# Fit 1: model Y ~ X1 (predictive)
X1_design = np.vstack([np.ones(n), X1]).T
beta_hat1, _, _, _ = np.linalg.lstsq(X1_design, np.log(Y + 1e-6), rcond=None)
mu_hat1 = np.exp(X1_design @ beta_hat1)

# Fit 2: model Y ~ X2 (irrelevant)
X2_design = np.vstack([np.ones(n), X2]).T
beta_hat2, _, _, _ = np.linalg.lstsq(X2_design, np.log(Y + 1e-6), rcond=None)
mu_hat2 = np.exp(X2_design @ beta_hat2)

# Compute residuals
resid1 = Y - mu_hat1
resid2 = Y - mu_hat2

# ==============================================================
# === PLOTTING FUNCTION ========================================
# ==============================================================

def plot_joint(X, Y, mu_hat, resid, title):
    df = pd.DataFrame({"X": X, "Y": Y, "mu": mu_hat, "resid": resid})

    g = sns.jointplot(data=df, x="X", y="Y", kind="scatter", height=6)
    ax = g.ax_joint

    # Plot fitted mean curve
    X_sorted = np.linspace(X.min(), X.max(), 200)
    mu_sorted = np.exp(np.polyval(np.polyfit(X, np.log(mu_hat + 1e-6), 1), X_sorted))
    ax.plot(X_sorted, mu_sorted, color="red", lw=2.5, label="Fitted mean")

    # Residuals as vertical lines
    for i in range(0, len(X), 10):  # every 10th point for clarity
        ax.plot([X[i], X[i]], [mu_hat[i], Y[i]], color="gray", alpha=0.6)

    ax.legend()
    ax.set_title(title)
    plt.show()


# ==============================================================
# === VISUALIZE ================================================
# ==============================================================

plot_joint(X1, Y, mu_hat1, resid1, "Predictive regressor X1 → Negative Binomial firing rate")
plot_joint(X2, Y, mu_hat2, resid2, "Irrelevant regressor X2 → No predictive power")


In [ ]:

# do the same for PCA and ensembles

